# ISIC 2024 - 圖片 CNN 訓練（EfficientNet-B0）

**目的**：用圖片訓練 CNN，產生每張訓練圖的 OOF 預測，給 LightGBM 當特徵（模型融合）。

**Kaggle 設定**：Accelerator 選 **GPU**；Internet **打開**（要下載 ImageNet 預訓練權重）。這個 notebook 不用提交。

**輸出**（存在 `/kaggle/working`，跑完按 Save Version）：
- `cnn_fold{0-4}.pt`：5 個 fold 的模型權重
- `oof_cnn.csv`：每張訓練圖的 OOF 預測

切 fold 的方式跟 LightGBM 完全一樣（StratifiedGroupKFold、seed 42、按 patient_id），融合時才不會資料洩漏。

In [ ]:
import glob, io, os, time
import numpy as np
import pandas as pd
import h5py
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

KAGGLE_INPUT = '/kaggle/input'

def find_dir(fname):
    # Kaggle 的掛載路徑會變（例如多一層 competitions/），所以用搜尋的找檔案所在資料夾
    for depth in range(1, 4):
        hits = sorted(glob.glob(os.path.join(KAGGLE_INPUT, *['*'] * depth, fname)),
                      key=lambda p: ('isic-2024-challenge' not in p, p))  # 優先選比賽本身的資料
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError(f'在 {KAGGLE_INPUT} 底下找不到 {fname}，請確認右側 Input 有加入')

DATA = find_dir('train-metadata.csv')
print('資料路徑:', DATA)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print('device:', DEVICE)

## 設定

In [ ]:
CFG = dict(
    img=128,          # 原圖約 130px，不需要放大太多
    epochs=4,
    bs=128,
    lr=3e-4,
    neg_ratio=20,     # 每個 epoch 重新抽良性，數量 = 惡性 × 20
    workers=4,
    pretrained=True,
    folds=5,
)

In [ ]:
def pauc_above_tpr(y_true, y_pred, min_tpr=0.80):
    v_gt = np.abs(np.asarray(y_true) - 1)
    v_pred = 1.0 - np.asarray(y_pred)
    max_fpr = 1 - min_tpr
    scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    return 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (scaled - 0.5)

## 資料集與模型

In [ ]:
class LesionDataset(Dataset):
    """從 hdf5 讀 JPEG bytes。每個 worker 各自開檔，避免 h5py 多程序問題。"""
    def __init__(self, h5_path, ids, targets=None, tfm=None):
        self.h5_path, self.ids, self.targets, self.tfm = h5_path, list(ids), targets, tfm
        self.h5 = None
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        if self.h5 is None:
            self.h5 = h5py.File(self.h5_path, 'r')
        img = Image.open(io.BytesIO(bytes(self.h5[self.ids[i]][()]))).convert('RGB')
        x = self.tfm(img)
        y = -1.0 if self.targets is None else float(self.targets[i])
        return x, torch.tensor(y, dtype=torch.float32)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tfm = T.Compose([
    T.Resize((CFG['img'], CFG['img'])),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(90),
    T.ColorJitter(0.2, 0.2, 0.1, 0.02),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tfm = T.Compose([T.Resize((CFG['img'], CFG['img'])), T.ToTensor(), T.Normalize(MEAN, STD)])

def make_model(pretrained):
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.efficientnet_b0(weights=weights)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    return m

@torch.no_grad()
def predict(model, h5_path, ids):
    ds = LesionDataset(h5_path, ids, tfm=eval_tfm)
    dl = DataLoader(ds, batch_size=CFG['bs'] * 2, shuffle=False, num_workers=CFG['workers'])
    model.eval()
    out = []
    for x, _ in dl:
        out.append(torch.sigmoid(model(x.to(DEVICE)).squeeze(1)).float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)

## 切 fold（跟 LightGBM 相同）

In [ ]:
train = pd.read_csv(f'{DATA}/train-metadata.csv', low_memory=False)
H5_TRAIN = f'{DATA}/train-image.hdf5'
sgkf = StratifiedGroupKFold(n_splits=CFG['folds'], shuffle=True, random_state=SEED)
folds = list(sgkf.split(train, train['target'], train['patient_id']))
print(train.shape, '惡性', int(train.target.sum()))

## 訓練

In [ ]:
oof = np.zeros(len(train))
rng = np.random.default_rng(SEED)
for fold, (tr, va) in enumerate(folds):
    t0 = time.time()
    d = train.iloc[tr]
    pos = d[d.target == 1]
    neg_pool = d[d.target == 0]
    model = make_model(CFG['pretrained']).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['epochs'])
    loss_fn = nn.BCEWithLogitsLoss()
    for ep in range(CFG['epochs']):
        n_neg = min(len(neg_pool), len(pos) * CFG['neg_ratio'])
        neg = neg_pool.iloc[rng.choice(len(neg_pool), n_neg, replace=False)]
        ep_df = pd.concat([pos, neg])
        dl = DataLoader(LesionDataset(H5_TRAIN, ep_df.isic_id, ep_df.target.values, train_tfm),
                        batch_size=CFG['bs'], shuffle=True, num_workers=CFG['workers'], drop_last=len(ep_df) > CFG['bs'])
        model.train()
        tot = 0.0
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(x).squeeze(1), y)
            loss.backward()
            opt.step()
            tot += loss.item() * len(y)
        sched.step()
        print(f'fold {fold} epoch {ep}: loss {tot / len(ep_df):.4f}')
    oof[va] = predict(model, H5_TRAIN, train.iloc[va].isic_id)
    torch.save(model.state_dict(), f'cnn_fold{fold}.pt')
    print(f'fold {fold}: pAUC = {pauc_above_tpr(train.iloc[va].target, oof[va]):.4f}  ({time.time() - t0:.0f}s)')

print(f'\nCNN OOF pAUC = {pauc_above_tpr(train.target, oof):.4f}')
pd.DataFrame({'isic_id': train.isic_id, 'cnn_pred': oof}).to_csv('oof_cnn.csv', index=False)